### **Importing Libraries**

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import kagglehub
from sklearn.cluster import KMeans


c:\Users\heppe\OneDrive\Documents\Python\K-means\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Retrieving Data from Kaggle**
https://www.kaggle.com/datasets/uom190346a/mental-health-research-dataset/data

In [6]:
# set path
path = kagglehub.dataset_download("uom190346a/mental-health-research-dataset")
# print out the path of downloaded files
print(path)

100%|██████████| 13.8k/13.8k [00:00<00:00, 1.88MB/s]

Extracting files...
C:\Users\heppe\.cache\kagglehub\datasets\uom190346a\mental-health-research-dataset\versions\1


You will need to retrieve the csv file from the stated path if you want to re-create this method. Otherwise I have uploaded the csv to this repository and will be accessing it from there for the rest of this project

### **Data Loading and Familiarisation**

In [7]:
# read csv
mh = pd.read_csv("Mental_Health_and_Lifestyle_Research.csv")

In [8]:
# head
mh.head()

,Person_ID,Age,Gender,Hours_of_Sleep,Stress_Level,Physical_Activity,Work_Hours_per_Day,Mental_Health_Status,Social_Interaction_Freq,Overall_Wellbeing_Score,Diet_Quality,Screen_Time_per_Day,Substance_Use,Has_Close_Friends,Physical_Health_Condition
0,1,25,Male,7,7,13,9,Mild Anxiety,Low,3,Excellent,8.8,NaN,True,NaN
1,2,34,Female,7,4,46,9,Depression,Moderate,5,Fair,5.1,Alcohol,True,Hypertension
2,3,31,Female,5,6,34,8,NaN,Moderate,4,Fair,6.6,Smoking,True,Obesity
3,4,29,Female,6,7,30,11,Depression,Moderate,2,Good,7.6,Alcohol,True,NaN
4,5,22,Female,7,5,10,7,NaN,Moderate,4,Very Good,6.5,Alcohol,True,Obesity


In [11]:
# information about data types and NAs
mh.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Person_ID                  1000 non-null   int64  
 1   Age                        1000 non-null   int64  
 2   Gender                     1000 non-null   str    
 3   Hours_of_Sleep             1000 non-null   int64  
 4   Stress_Level               1000 non-null   int64  
 5   Physical_Activity          1000 non-null   int64  
 6   Work_Hours_per_Day         1000 non-null   int64  
 7   Mental_Health_Status       358 non-null    str    
 8   Social_Interaction_Freq    1000 non-null   str    
 9   Overall_Wellbeing_Score    1000 non-null   int64  
 10  Diet_Quality               1000 non-null   str    
 11  Screen_Time_per_Day        1000 non-null   float64
 12  Substance_Use              621 non-null    str    
 13  Has_Close_Friends          1000 non-null   bool   
 14  Phys

In [12]:
# look for data ranges
mh.describe()

,Person_ID,Age,Hours_of_Sleep,Stress_Level,Physical_Activity,Work_Hours_per_Day,Overall_Wellbeing_Score,Screen_Time_per_Day
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,27.076000,6.950000,5.46900,28.954000,7.994000,4.094000,7.088500
std,288.819436,4.578995,1.127285,2.03892,18.024671,1.412076,2.131128,1.984433
min,1.000000,20.000000,5.000000,1.00000,0.000000,5.000000,1.000000,1.000000
25%,250.750000,23.000000,6.000000,4.00000,15.000000,7.000000,2.000000,5.800000
50%,500.500000,27.000000,7.000000,5.00000,29.000000,8.000000,4.000000,7.100000
75%,750.250000,31.000000,8.000000,7.00000,41.000000,9.000000,6.000000,8.400000
max,1000.000000,35.000000,9.000000,10.00000,93.000000,11.000000,10.000000,12.000000


### **Data Cleaning**
There are a lot of `NaN` entries in some of the categorical columns:
- `Mental_Health_Status`
- `Substance_Use`
- `Physical_Health_Status`

Looking at the [Kaggle documentation](https://www.kaggle.com/datasets/uom190346a/mental-health-research-dataset/data), this would appear to be pandas interpreting `"None"` as `NaN` - going to replace `NaN` with a string.

In [19]:
# replace NaN
mh.fillna("None", inplace=True)

# check it worked
mh.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Person_ID                  1000 non-null   int64  
 1   Age                        1000 non-null   int64  
 2   Gender                     1000 non-null   str    
 3   Hours_of_Sleep             1000 non-null   int64  
 4   Stress_Level               1000 non-null   int64  
 5   Physical_Activity          1000 non-null   int64  
 6   Work_Hours_per_Day         1000 non-null   int64  
 7   Mental_Health_Status       1000 non-null   str    
 8   Social_Interaction_Freq    1000 non-null   str    
 9   Overall_Wellbeing_Score    1000 non-null   int64  
 10  Diet_Quality               1000 non-null   str    
 11  Screen_Time_per_Day        1000 non-null   float64
 12  Substance_Use              1000 non-null   str    
 13  Has_Close_Friends          1000 non-null   bool   
 14  Phys

As per the `describe()` output, all of the continuous variables appear to have a fairly 

### **Pre-Processing**

One-Hot Encoding categorical variables